In [ ]:
import os
import json
import time
import logging
from pathlib import Path
from google import genai
from google.genai import types
from tqdm import tqdm

# ─── CONFIG ──────────────────────────────────────────────────────────────────

MODEL               = "gemini-2.5-pro"
SAMPLES_PER_CALL    = 50
CALLS_PER_BAND      = 4
BANDS_PER_TARGET    = 5
TARGETS_PER_EMOTION = 4
DELAY_BETWEEN_CALLS = 1.0
MAX_RETRIES         = 3
OUTPUT_DIR          = Path("/content/drive/MyDrive/emo/emotion_data")

EMOTIONS = ["anger", "sadness", "happiness", "fear", "disgust", "surprise"]

TARGETS = ["you", "other", "self", "situation"]

TARGET_DEFINITIONS = {
    "you":       "directed at the AI assistant",
    "other":     "directed at a third person or group",
    "self":      "directed at themselves",
    "situation": "directed at an event, circumstance, or environment — including pure venting with no specific human target "
}

INTENSITY_BANDS = [
    (0.10, 0.27),
    (0.28, 0.44),
    (0.45, 0.60),
    (0.61, 0.77),
    (0.78, 1.00),
]

# ─── LOGGING ─────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(OUTPUT_DIR / "generation.log"),
    ],
)
log = logging.getLogger(__name__)

# ─── PROMPTS ─────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """\
You are a dataset generator for an emotion classifier.
Return ONLY a valid JSON array. No explanation, no markdown, no preamble."""


def build_prompt(emotion: str, target: str, low: float, high: float) -> str:
    return f"""\
Generate exactly {SAMPLES_PER_CALL} samples.

Fixed for this batch:
  emotion   : {emotion}
  target    : {target}  ({TARGET_DEFINITIONS[target]})
  intensity : between {low:.2f} and {high:.2f}

Schema for each sample:
{{
  "text":      "1-2 casual sentences like texting a friend in 10-20 words",
  "emotion":   "{emotion}",
  "target":    "{target}",
  "intensity": <float between {low:.2f} and {high:.2f}>
}}

Rules:
1. Spread intensity uniformly across [{low:.2f}, {high:.2f}].
2. Every text must be distinct — no repeated phrasing or structure.
3. Casual tone like human conversation.
4. Vary life context: relationships, work, health, study, family, daily life, exams, etc.
5. Return ONLY the JSON array. Nothing else."""


# ─── GEMINI CALL ─────────────────────────────────────────────────────────────

def call_gemini(client: genai.Client, prompt: str) -> list:
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            temperature=1.0,
            max_output_tokens=8192,
        ),
    )
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = "\n".join(raw.split("\n")[1:])
    if raw.endswith("```"):
        raw = raw[: raw.rfind("```")]
    return json.loads(raw.strip())


# ─── VALIDATOR ───────────────────────────────────────────────────────────────

def validate_batch(samples: list, emotion: str, target: str,
                   low: float, high: float):
    clean  = []
    errors = []

    for i, s in enumerate(samples):
        tag = f"[idx_{i}]"

        if not isinstance(s.get("text"), str) or len(s["text"].strip()) < 5:
            errors.append(f"{tag} text missing or too short")
            continue
        if s.get("emotion") != emotion:
            errors.append(f"{tag} wrong emotion: {s.get('emotion')}")
            continue
        if s.get("target") != target:
            errors.append(f"{tag} wrong target: {s.get('target')}")
            continue

        intensity = s.get("intensity")
        if not isinstance(intensity, (int, float)):
            errors.append(f"{tag} intensity not numeric: {intensity}")
            continue
        intensity = float(intensity)
        if not (low - 0.02 <= intensity <= high + 0.02):
            errors.append(
                f"{tag} intensity {intensity:.2f} outside [{low:.2f},{high:.2f}]")
            continue

        clean.append({
            "text":      s["text"].strip(),
            "emotion":   emotion,
            "target":    target,
            "intensity": round(intensity, 4),
        })

    return clean, errors


def assign_ids(samples: list) -> list:
    for i, s in enumerate(samples, start=1):
        s["id"] = i
    return samples


# ─── PROGRESS TRACKING ───────────────────────────────────────────────────────

def progress_path(emotion: str) -> Path:
    return OUTPUT_DIR / f"{emotion}_progress.json"


def load_progress(emotion: str) -> dict:
    p = progress_path(emotion)
    return json.loads(p.read_text()) if p.exists() else {}


def save_progress(emotion: str, progress: dict):
    progress_path(emotion).write_text(json.dumps(progress, indent=2))


def call_key(target: str, band_idx: int, call_idx: int) -> str:
    return f"{target}_{band_idx}_{call_idx}"


# ─── GENERATION ──────────────────────────────────────────────────────────────

def generate_emotion(client: genai.Client, emotion: str):
    """
    Generates 4000 samples for one emotion (1000 per target × 4 targets).
    Writes incrementally and resumes from checkpoint on interrupt.
    """
    log.info(f"{'='*60}")
    log.info(f"  Emotion: {emotion.upper()}")
    log.info(f"{'='*60}")

    out_file = OUTPUT_DIR / f"{emotion}.json"
    raw_dir  = OUTPUT_DIR / "raw" / emotion
    raw_dir.mkdir(parents=True, exist_ok=True)

    existing_samples = []
    completed_keys   = set()

    if out_file.exists():
        existing_data    = json.loads(out_file.read_text())
        existing_samples = [{k: v for k, v in s.items() if k != "id"}
                            for s in existing_data]
        log.info(f"  Found {len(existing_samples)} existing samples")
        progress       = load_progress(emotion)
        completed_keys = set(progress.keys())
        log.info(f"  Completed {len(completed_keys)} API calls")

    new_samples  = []
    total_calls  = TARGETS_PER_EMOTION * BANDS_PER_TARGET * CALLS_PER_BAND
    total_errors = 0

    call_pbar = tqdm(total=total_calls, desc=f"{emotion} API calls",
                     unit="call", position=0, leave=True)
    call_pbar.update(len(completed_keys))

    for target in TARGETS:
        target_count = 0

        for band_idx, (low, high) in enumerate(INTENSITY_BANDS):
            for call_idx in range(CALLS_PER_BAND):
                key = call_key(target, band_idx, call_idx)

                if key in completed_keys:
                    continue

                call_pbar.set_description(
                    f"{emotion} [{target}] {low:.2f}-{high:.2f}")

                prompt = build_prompt(emotion, target, low, high)
                batch  = []

                for attempt in range(1, MAX_RETRIES + 1):
                    try:
                        raw_batch = call_gemini(client, prompt)
                        clean, errs = validate_batch(
                            raw_batch, emotion, target, low, high)

                        if errs:
                            log.warning(f"    {len(errs)} validation errors:")
                            for e in errs[:3]:
                                log.warning(f"      {e}")
                            if len(errs) > 3:
                                log.warning(f"      ...+{len(errs)-3} more")
                        total_errors += len(errs)

                        (raw_dir / f"{target}_b{band_idx}_c{call_idx}.json").write_text(
                            json.dumps(raw_batch, indent=2, ensure_ascii=False))

                        batch = clean
                        break

                    except json.JSONDecodeError as e:
                        log.error(f"    Attempt {attempt}: JSON parse error — {e}")
                        if attempt < MAX_RETRIES:
                            time.sleep(DELAY_BETWEEN_CALLS * 2)
                    except Exception as e:
                        log.error(f"    Attempt {attempt}: API error — {e}")
                        if attempt < MAX_RETRIES:
                            time.sleep(DELAY_BETWEEN_CALLS * 3)

                if batch:
                    new_samples.extend(batch)
                    target_count += len(batch)

                    all_current = existing_samples + new_samples
                    out_file.write_text(
                        json.dumps(all_current, indent=2, ensure_ascii=False))

                    progress = load_progress(emotion)
                    progress[key] = {
                        "completed": True,
                        "samples":   len(batch),
                        "timestamp": time.time(),
                    }
                    save_progress(emotion, progress)
                    completed_keys.add(key)

                call_pbar.update(1)

                if call_pbar.n < total_calls:
                    time.sleep(DELAY_BETWEEN_CALLS)

        log.info(f"  Target '{target}': {target_count} new samples")

    call_pbar.close()

    all_samples = existing_samples + new_samples
    if all_samples:
        all_samples = assign_ids(all_samples)
        out_file.write_text(
            json.dumps(all_samples, indent=2, ensure_ascii=False))
        progress_path(emotion).unlink(missing_ok=True)
        log.info(f"\n  {emotion.upper()} COMPLETE — "
                 f"{len(all_samples)} samples, {total_errors} errors dropped")
        log.info(f"  Saved: {out_file}\n")
    else:
        log.error(f"No samples generated for {emotion}")

    return all_samples


# ─── REPORT ──────────────────────────────────────────────────────────────────

def print_report(emotion: str, samples: list):
    from collections import Counter

    if not samples:
        print(f"\nNo samples for {emotion}")
        return

    target_counts = Counter(s["target"] for s in samples)
    intensities   = [float(s["intensity"]) for s in samples
                     if isinstance(s.get("intensity"), (int, float))]

    print(f"\n{'='*50}")
    print(f"  {emotion.upper()}  ({len(samples)} samples)")
    print(f"{'='*50}")
    print("  Target distribution:")
    for t in TARGETS:
        n    = target_counts.get(t, 0)
        pct  = n / len(samples) * 100 if samples else 0
        bar  = "█" * (n // 25)
        flag = " ⚠ LOW" if n < 900 else ""
        print(f"    {t:<12} {n:>4}  ({pct:4.1f}%)  {bar}{flag}")

    if intensities:
        print(f"\n  Intensity  min={min(intensities):.2f}  "
              f"max={max(intensities):.2f}  "
              f"avg={sum(intensities)/len(intensities):.2f}")
        print("  Bands:")
        for low, high in INTENSITY_BANDS:
            n    = sum(1 for x in intensities if low - 0.01 <= x <= high + 0.01)
            bar  = "█" * (n // 25)
            flag = " ⚠ LOW" if n < 150 else ""
            print(f"    [{low:.2f}-{high:.2f}]  {n:>4}  {bar}{flag}")
    print()


# ─── ENTRY POINT ─────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("  Emotion Dataset Generator")
    print("=" * 60)
    print(f"\n  Emotions  : {EMOTIONS}")
    print(f"  Targets   : {TARGETS}")
    print(f"  Per target: {BANDS_PER_TARGET} bands × {CALLS_PER_BAND} calls "
          f"× {SAMPLES_PER_CALL} samples = "
          f"{BANDS_PER_TARGET * CALLS_PER_BAND * SAMPLES_PER_CALL}")
    print(f"  Per emotion: "
          f"{TARGETS_PER_EMOTION * BANDS_PER_TARGET * CALLS_PER_BAND * SAMPLES_PER_CALL} samples")
    print()

    api_key = ''
    if not api_key:
        raise EnvironmentError("API key is required")

    client = genai.Client(api_key=api_key)
    log.info(f"Model  : {MODEL}")
    log.info(f"Output : {OUTPUT_DIR.resolve()}")
    log.info("Checkpoint system ENABLED — resumes if interrupted")

    grand_total = 0
    for emotion in EMOTIONS:
        samples = generate_emotion(client, emotion)
        print_report(emotion, samples)
        grand_total += len(samples)
        if emotion != EMOTIONS[-1]:
            time.sleep(DELAY_BETWEEN_CALLS * 2)

    log.info(f"\n{'='*60}")
    log.info(f"  ALL DONE — {grand_total} total samples")
    log.info(f"  Files in: {OUTPUT_DIR.resolve()}")
    log.info(f"{'='*60}")


if __name__ == "__main__":
    main()

  Emotion Dataset Generator

  Emotions  : ['anger', 'sadness', 'happiness', 'fear', 'disgust', 'surprise']
  Targets   : ['you', 'other', 'self', 'situation']
  Per target: 5 bands × 4 calls × 50 samples = 1000
  Per emotion: 4000 samples



anger [self] 0.61-0.77:  66%|██████▋   | 53/80 [37:26<16:46, 37.29s/call]ERROR:__main__:    Attempt 1: JSON parse error — Unterminated string starting at: line 261 column 13 (char 7684)
ERROR:__main__:    Attempt 2: JSON parse error — Unterminated string starting at: line 273 column 13 (char 7515)
anger [situation] 0.78-1.00: 100%|██████████| 80/80 [59:52<00:00, 44.90s/call]



  ANGER  (4001 samples)
  Target distribution:
    you          1001  (25.0%)  ████████████████████████████████████████
    other        1000  (25.0%)  ████████████████████████████████████████
    self         1000  (25.0%)  ████████████████████████████████████████
    situation    1000  (25.0%)  ████████████████████████████████████████

  Intensity  min=0.10  max=1.00  avg=0.54
  Bands:
    [0.10-0.27]   822  ████████████████████████████████
    [0.28-0.44]   868  ██████████████████████████████████
    [0.45-0.60]   865  ██████████████████████████████████
    [0.61-0.77]   851  ██████████████████████████████████
    [0.78-1.00]   835  █████████████████████████████████



sadness [situation] 0.78-1.00: 100%|██████████| 80/80 [1:02:22<00:00, 46.79s/call]



  SADNESS  (4000 samples)
  Target distribution:
    you          1000  (25.0%)  ████████████████████████████████████████
    other        1000  (25.0%)  ████████████████████████████████████████
    self         1000  (25.0%)  ████████████████████████████████████████
    situation    1000  (25.0%)  ████████████████████████████████████████

  Intensity  min=0.10  max=1.00  avg=0.54
  Bands:
    [0.10-0.27]   821  ████████████████████████████████
    [0.28-0.44]   861  ██████████████████████████████████
    [0.45-0.60]   854  ██████████████████████████████████
    [0.61-0.77]   839  █████████████████████████████████
    [0.78-1.00]   835  █████████████████████████████████



...


  FEAR  (4001 samples)
  Target distribution:
    you          1000  (25.0%)  ████████████████████████████████████████
    other        1000  (25.0%)  ████████████████████████████████████████
    self         1000  (25.0%)  ████████████████████████████████████████
    situation    1001  (25.0%)  ████████████████████████████████████████

  Intensity  min=0.10  max=1.00  avg=0.54
  Bands:
    [0.10-0.27]   821  ████████████████████████████████
    [0.28-0.44]   865  ██████████████████████████████████
    [0.45-0.60]   862  ██████████████████████████████████
    [0.61-0.77]   845  █████████████████████████████████
    [0.78-1.00]   842  █████████████████████████████████



disgust [situation] 0.78-1.00: 100%|██████████| 80/80 [59:05<00:00, 44.32s/call]



  DISGUST  (3999 samples)
  Target distribution:
    you          1000  (25.0%)  ████████████████████████████████████████
    other         999  (25.0%)  ███████████████████████████████████████
    self         1000  (25.0%)  ████████████████████████████████████████
    situation    1000  (25.0%)  ████████████████████████████████████████

  Intensity  min=0.10  max=1.00  avg=0.53
  Bands:
    [0.10-0.27]   825  █████████████████████████████████
    [0.28-0.44]   851  ██████████████████████████████████
    [0.45-0.60]   858  ██████████████████████████████████
    [0.61-0.77]   848  █████████████████████████████████
    [0.78-1.00]   837  █████████████████████████████████



surprise [situation] 0.78-1.00: 100%|██████████| 80/80 [54:44<00:00, 41.05s/call]


  SURPRISE  (3999 samples)
  Target distribution:
    you          1000  (25.0%)  ████████████████████████████████████████
    other        1000  (25.0%)  ████████████████████████████████████████
    self          999  (25.0%)  ███████████████████████████████████████
    situation    1000  (25.0%)  ████████████████████████████████████████

  Intensity  min=0.10  max=1.00  avg=0.53
  Bands:
    [0.10-0.27]   823  ████████████████████████████████
    [0.28-0.44]   853  ██████████████████████████████████
    [0.45-0.60]   856  ██████████████████████████████████
    [0.61-0.77]   836  █████████████████████████████████
    [0.78-1.00]   840  █████████████████████████████████



In [2]:
import json
import random
import argparse
from pathlib import Path
from collections import defaultdict


DATA_DIR = "/content/drive/MyDrive/emo/emotion_data"
OUTPUT_DIR = "/content/drive/MyDrive/emo/emotion_data/splits"
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
SEED = 42


# ─── CONSTANTS ───────────────────────────────────────────────────────────────

EMOTIONS = ["anger", "sadness", "happiness", "fear", "disgust", "surprise"]
TARGETS  = ["you", "other", "self", "situation"]

# 6 intensity bands — same as generation script
INTENSITY_BANDS = [
    (0.10, 0.17),
    (0.18, 0.34),
    (0.35, 0.50),
    (0.51, 0.67),
    (0.68, 0.84),
    (0.85, 1.00),
]

def get_intensity_band(intensity: float) -> int:
    for i, (low, high) in enumerate(INTENSITY_BANDS):
        if low <= intensity <= high + 0.01:
            return i
    return len(INTENSITY_BANDS) - 1


def split_group(samples: list, train_r: float, val_r: float, rng: random.Random):
    rng.shuffle(samples)
    n = len(samples)

    if n < 3:
        return samples, [], []

    n_train = max(1, int(n * train_r))
    n_val   = max(1, int(n * val_r))
    n_train = min(n_train, n - 2)
    n_val   = min(n_val,   n - n_train - 1)

    return samples[:n_train], samples[n_train:n_train + n_val], samples[n_train + n_val:]


def split_dataset(data_dir: str, output_dir: str,
                  train_ratio: float = 0.70,
                  val_ratio:   float = 0.15,
                  seed:        int   = 42):

    rng = random.Random(seed)
    data_dir   = Path(data_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_train, all_val, all_test = [], [], []

    for emotion in EMOTIONS:
        path = data_dir / f"{emotion}.json"
        if not path.exists():
            print(f"  WARNING: {path} not found, skipping")
            continue

        samples = json.loads(path.read_text())
        print(f"\n  {emotion.upper()} — {len(samples)} samples")

        # Group by target + intensity_band
        groups = defaultdict(list)
        for s in samples:
            band = get_intensity_band(float(s["intensity"]))
            groups[(s["target"], band)].append(s)

        e_train, e_val, e_test = [], [], []

        for (target, band), group in sorted(groups.items()):
            t, v, te = split_group(group, train_ratio, val_ratio, rng)
            e_train.extend(t)
            e_val.extend(v)
            e_test.extend(te)

            low, high = INTENSITY_BANDS[band]
            print(f"    target={target:<12} band=[{low:.2f}-{high:.2f}]  "
                  f"n={len(group):>3}  "
                  f"train={len(t):>3}  val={len(v):>3}  test={len(te):>3}")

        print(f"  Total: train={len(e_train)}  val={len(e_val)}  test={len(e_test)}")
        all_train.extend(e_train)
        all_val.extend(e_val)
        all_test.extend(e_test)

    # Shuffle combined splits
    rng.shuffle(all_train)
    rng.shuffle(all_val)
    rng.shuffle(all_test)

    # Assign clean integer IDs per split
    for i, s in enumerate(all_train, 1): s["id"] = i
    for i, s in enumerate(all_val,   1): s["id"] = i
    for i, s in enumerate(all_test,  1): s["id"] = i

    # Save
    (output_dir / "train.json").write_text(json.dumps(all_train, indent=2, ensure_ascii=False))
    (output_dir / "val.json").write_text(json.dumps(all_val,   indent=2, ensure_ascii=False))
    (output_dir / "test.json").write_text(json.dumps(all_test,  indent=2, ensure_ascii=False))

    print(f"\n{'='*50}")
    print(f"  FINAL SPLIT")
    print(f"  Train : {len(all_train):>5}")
    print(f"  Val   : {len(all_val):>5}")
    print(f"  Test  : {len(all_test):>5}")
    print(f"  Total : {len(all_train)+len(all_val)+len(all_test):>5}")
    print(f"{'='*50}")

    print("\n  Coverage check — val:")
    _check_coverage(all_val)
    print("\n  Coverage check — test:")
    _check_coverage(all_test)

    print(f"\n  Saved to {output_dir.resolve()}")


def _check_coverage(samples: list):
    emotions_found = set(s["emotion"] for s in samples)
    targets_found  = set(s["target"]  for s in samples)
    bands_found    = set(get_intensity_band(float(s["intensity"])) for s in samples)

    missing_e = set(EMOTIONS) - emotions_found
    missing_t = set(TARGETS)  - targets_found
    missing_b = set(range(len(INTENSITY_BANDS))) - bands_found

    print(f"    Emotions  : {sorted(emotions_found)}"
          + (f"  MISSING {missing_e}" if missing_e else "  OK"))
    print(f"    Targets   : {sorted(targets_found)}"
          + (f"  MISSING {missing_t}" if missing_t else "  OK"))
    print(f"    Int bands : {sorted(bands_found)}"
          + (f"  MISSING {missing_b}" if missing_b else "  OK"))


def main():
    split_dataset(DATA_DIR, OUTPUT_DIR, TRAIN_RATIO, VAL_RATIO, SEED)


if __name__ == "__main__":
    main()


  ANGER — 4000 samples
    target=other        band=[0.10-0.17]  n= 84  train= 58  val= 12  test= 14
    target=other        band=[0.18-0.34]  n=194  train=135  val= 29  test= 30
    target=other        band=[0.35-0.50]  n=201  train=140  val= 30  test= 31
    target=other        band=[0.51-0.67]  n=205  train=143  val= 30  test= 32
    target=other        band=[0.68-0.84]  n=169  train=118  val= 25  test= 26
    target=other        band=[0.85-1.00]  n=147  train=102  val= 22  test= 23
    target=self         band=[0.10-0.17]  n= 87  train= 60  val= 13  test= 14
    target=self         band=[0.18-0.34]  n=199  train=139  val= 29  test= 31
    target=self         band=[0.35-0.50]  n=194  train=135  val= 29  test= 30
    target=self         band=[0.51-0.67]  n=204  train=142  val= 30  test= 32
    target=self         band=[0.68-0.84]  n=173  train=121  val= 25  test= 27
    target=self         band=[0.85-1.00]  n=143  train=100  val= 21  test= 22
    target=situation    band=[0.10-0.17]

In [ ]:
import json
import time
import logging
from pathlib import Path
from google import genai
from google.genai import types
from tqdm import tqdm

# ─── CONFIG ──────────────────────────────────────────────────────────────────

MODEL                    = "gemini-2.5-pro"
SAMPLES_PER_CALL         = 40
CALLS_PER_DIRECTION      = 5     # 5 calls × 40 = 200 per direction
                                 # 2 directions → 400 total boundary samples
DELAY_BETWEEN_CALLS      = 1.0
MAX_RETRIES              = 3
OUTPUT_DIR               = Path("/content/drive/MyDrive/emo/emotion_data")
OUTPUT_FILE              = OUTPUT_DIR / "boundary_anger_sadness.json"
PROGRESS_FILE            = OUTPUT_DIR / "boundary_progress.json"

VALID_TARGETS = ["you", "other", "self", "situation"]

# Two directions: each produces CALLS_PER_DIRECTION × SAMPLES_PER_CALL samples
# Direction key → (true_emotion, confusable_emotion)
DIRECTIONS = {
    "anger_looks_sad":    ("anger",   "sadness"),
    "sadness_looks_angry": ("sadness", "anger"),
}

# ─── LOGGING ─────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(OUTPUT_DIR / "boundary_generation.log"),
    ],
)
log = logging.getLogger(__name__)

# ─── PROMPTS ─────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """\
You are a dataset generator for an emotion classifier.
Return ONLY a valid JSON array. No explanation, no markdown, no preamble."""


def build_boundary_prompt(true_emotion: str, confusable_emotion: str) -> str:
    """
    Generates hard-boundary samples: texts whose TRUE label is `true_emotion`
    but that strongly resemble `confusable_emotion` on the surface.
    These edge cases help the model learn the fine distinction between the two.
    """
    # Direction-specific guidance so the LLM knows exactly what to aim for
    if true_emotion == "anger" and confusable_emotion == "sadness":
        guidance = """\
The text should carry an undercurrent of FURY / betrayal / injustice, but the
surface wording uses loss-like or hurt-like language that a naive reader might
call sadness.  The speaker is not grieving — they are enraged.

Good examples:
  - "I can't believe they just left without saying a single word to me."
      → sounds like hurt/loss but speaker is actually furious at the betrayal
  - "Five years and they couldn't even bother to reply to my message."
      → sounds dejected but the emotion is simmering anger at disrespect
  - "Wow, so I just don't matter to anyone here I guess."
      → passive phrasing hides real anger"""

    else:  # sadness looks like anger
        guidance = """\
The text should carry an undercurrent of GRIEF / defeat / heartbreak, but the
surface wording uses frustrated or accusatory language that a naive reader might
call anger.  The speaker is not enraged — they are heartbroken or despairing.

Good examples:
  - "I worked so hard for nothing, nothing ever changes no matter what I do."
      → sounds like frustration/anger but speaker is actually defeated/hopeless
  - "Why does everyone always end up leaving eventually."
      → sounds accusatory but emotion is deep sadness and resignation
  - "Great, another thing I messed up, as expected."
      → sounds bitter but the real emotion is self-directed grief/despair"""

    return f"""\
Generate exactly {SAMPLES_PER_CALL} HARD-BOUNDARY samples.

TRUE emotion  : {true_emotion}
LOOKS LIKE    : {confusable_emotion}

{guidance}

Schema for each sample:
{{
  "text":          "1-2 casual sentences, 10-20 words, conversational tone",
  "emotion":       "{true_emotion}",
  "target":        <one of: "you", "other", "self", "situation">,
  "intensity":     <float 0.30 – 0.85, spread uniformly>,
  "boundary_note": "≤15 words: why this is {true_emotion} not {confusable_emotion}"
}}

Rules:
1. TRUE label is ALWAYS "{true_emotion}" — confusability is surface-level only.
2. Mix targets across the batch: roughly equal split of you/other/self/situation.
3. Vary life contexts: relationships, work, health, exams, family, grief, daily frustration.
4. Every text must be distinct — no repeated phrasing or structure.
5. Return ONLY the JSON array. Nothing else."""


# ─── GEMINI CALL ─────────────────────────────────────────────────────────────

def call_gemini(client: genai.Client, prompt: str) -> list:
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            temperature=1.0,
            max_output_tokens=8192,
        ),
    )
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = "\n".join(raw.split("\n")[1:])
    if raw.endswith("```"):
        raw = raw[: raw.rfind("```")]
    return json.loads(raw.strip())


# ─── VALIDATOR ───────────────────────────────────────────────────────────────

def validate_batch(samples: list, true_emotion: str) -> tuple[list, list]:
    clean  = []
    errors = []

    for i, s in enumerate(samples):
        tag = f"[idx_{i}]"

        if not isinstance(s.get("text"), str) or len(s["text"].strip()) < 5:
            errors.append(f"{tag} text missing or too short")
            continue

        if s.get("emotion") != true_emotion:
            errors.append(
                f"{tag} wrong emotion: {s.get('emotion')!r} (expected {true_emotion!r})")
            continue

        if s.get("target") not in VALID_TARGETS:
            errors.append(f"{tag} invalid target: {s.get('target')!r}")
            continue

        intensity = s.get("intensity")
        if not isinstance(intensity, (int, float)):
            errors.append(f"{tag} intensity not numeric: {intensity}")
            continue
        intensity = float(intensity)
        if not (0.28 <= intensity <= 0.87):
            errors.append(f"{tag} intensity {intensity:.2f} out of range [0.28, 0.87]")
            continue

        clean.append({
            "text":          s["text"].strip(),
            "emotion":       true_emotion,
            "target":        s["target"],
            "intensity":     round(intensity, 4),
            "boundary_note": str(s.get("boundary_note", "")).strip(),
            "is_boundary":   True,   # flag for optional loss-weighting during training
        })

    return clean, errors


def assign_ids(samples: list) -> list:
    for i, s in enumerate(samples, start=1):
        s["id"] = i
    return samples


# ─── PROGRESS TRACKING ───────────────────────────────────────────────────────

def load_progress() -> dict:
    return json.loads(PROGRESS_FILE.read_text()) if PROGRESS_FILE.exists() else {}


def save_progress(progress: dict):
    PROGRESS_FILE.write_text(json.dumps(progress, indent=2))


def call_key(direction: str, call_idx: int) -> str:
    return f"{direction}_{call_idx}"


# ─── GENERATION ──────────────────────────────────────────────────────────────

def generate_boundary_samples(client: genai.Client) -> list:
    """
    Generates ~400 hard-boundary anger/sadness samples:
      • 200 anger samples that look like sadness
      • 200 sadness samples that look like anger
    Writes incrementally and resumes from checkpoint on interrupt.
    Output: boundary_anger_sadness.json
    """
    log.info(f"{'='*60}")
    log.info("  HARD-BOUNDARY: anger ↔ sadness")
    log.info(f"  {CALLS_PER_DIRECTION} calls × {SAMPLES_PER_CALL} samples "
             f"× 2 directions = ~{CALLS_PER_DIRECTION * SAMPLES_PER_CALL * 2} samples")
    log.info(f"{'='*60}")

    raw_dir = OUTPUT_DIR / "raw" / "boundary"
    raw_dir.mkdir(parents=True, exist_ok=True)

    # Load existing samples and progress
    existing = []
    if OUTPUT_FILE.exists():
        existing = json.loads(OUTPUT_FILE.read_text())
        existing = [{k: v for k, v in s.items() if k != "id"} for s in existing]
        log.info(f"  Resuming — found {len(existing)} existing samples")

    progress       = load_progress()
    completed_keys = set(progress.keys())
    total_calls    = len(DIRECTIONS) * CALLS_PER_DIRECTION

    log.info(f"  Completed calls so far: {len(completed_keys)} / {total_calls}")

    new_samples  = []
    total_errors = 0

    pbar = tqdm(total=total_calls, desc="boundary calls",
                unit="call", leave=True)
    pbar.update(len(completed_keys))

    for direction_key, (true_emo, confusable_emo) in DIRECTIONS.items():
        log.info(f"\n  Direction: {true_emo!r} (looks like {confusable_emo!r})")
        direction_count = 0

        for call_idx in range(CALLS_PER_DIRECTION):
            key = call_key(direction_key, call_idx)

            if key in completed_keys:
                continue

            pbar.set_description(
                f"boundary [{true_emo}→{confusable_emo}] call {call_idx+1}")

            prompt = build_boundary_prompt(true_emo, confusable_emo)
            batch  = []

            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    raw_batch = call_gemini(client, prompt)
                    clean, errs = validate_batch(raw_batch, true_emo)

                    if errs:
                        log.warning(f"    {len(errs)} validation errors:")
                        for e in errs[:3]:
                            log.warning(f"      {e}")
                        if len(errs) > 3:
                            log.warning(f"      ...+{len(errs)-3} more")
                    total_errors += len(errs)

                    fname = f"{direction_key}_c{call_idx}.json"
                    (raw_dir / fname).write_text(
                        json.dumps(raw_batch, indent=2, ensure_ascii=False))

                    batch = clean
                    break

                except json.JSONDecodeError as e:
                    log.error(f"    Attempt {attempt}: JSON parse error — {e}")
                    if attempt < MAX_RETRIES:
                        time.sleep(DELAY_BETWEEN_CALLS * 2)
                except Exception as e:
                    log.error(f"    Attempt {attempt}: API error — {e}")
                    if attempt < MAX_RETRIES:
                        time.sleep(DELAY_BETWEEN_CALLS * 3)

            if batch:
                new_samples.extend(batch)
                direction_count += len(batch)

                # Incremental write
                all_current = existing + new_samples
                OUTPUT_FILE.write_text(
                    json.dumps(all_current, indent=2, ensure_ascii=False))

                progress[key] = {
                    "completed": True,
                    "samples":   len(batch),
                    "timestamp": time.time(),
                }
                save_progress(progress)
                completed_keys.add(key)

            pbar.update(1)
            time.sleep(DELAY_BETWEEN_CALLS)

        log.info(f"  {true_emo!r} direction: {direction_count} new samples")

    pbar.close()

    # Final write with clean sequential IDs
    all_samples = existing + new_samples
    if all_samples:
        all_samples = assign_ids(all_samples)
        OUTPUT_FILE.write_text(
            json.dumps(all_samples, indent=2, ensure_ascii=False))
        PROGRESS_FILE.unlink(missing_ok=True)   # clean up checkpoint
    else:
        log.error("No boundary samples generated")

    return all_samples


# ─── REPORT ──────────────────────────────────────────────────────────────────

def print_report(samples: list):
    from collections import Counter

    if not samples:
        print("\nNo boundary samples to report.")
        return

    emo_counts    = Counter(s["emotion"] for s in samples)
    target_counts = Counter(s["target"]  for s in samples)
    intensities   = [float(s["intensity"]) for s in samples
                     if isinstance(s.get("intensity"), (int, float))]

    print(f"\n{'='*55}")
    print(f"  BOUNDARY SAMPLES — anger ↔ sadness  ({len(samples)} total)")
    print(f"{'='*55}")

    print("\n  Emotion split:")
    for emo in ["anger", "sadness"]:
        n   = emo_counts.get(emo, 0)
        pct = n / len(samples) * 100
        bar = "█" * (n // 5)
        print(f"    {emo:<10} {n:>3}  ({pct:4.1f}%)  {bar}")

    print("\n  Target distribution:")
    for t in ["you", "other", "self", "situation"]:
        n   = target_counts.get(t, 0)
        pct = n / len(samples) * 100
        bar = "█" * (n // 5)
        print(f"    {t:<12} {n:>3}  ({pct:4.1f}%)  {bar}")

    if intensities:
        print(f"\n  Intensity  min={min(intensities):.2f}  "
              f"max={max(intensities):.2f}  "
              f"avg={sum(intensities)/len(intensities):.2f}")

    print(f"\n  Saved to : {OUTPUT_FILE}")
    print()
    print("  ► Merge into train.json before re-training.")
    print("    The 'is_boundary' flag lets you apply higher sample")
    print("    weights in your loss function if desired (e.g. 1.5×–2×).")
    print()


# ─── ENTRY POINT ─────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("  Anger / Sadness Hard-Boundary Generator")
    print("=" * 60)
    print(f"\n  Directions : {list(DIRECTIONS.keys())}")
    print(f"  Per dir    : {CALLS_PER_DIRECTION} calls × {SAMPLES_PER_CALL} samples "
          f"= {CALLS_PER_DIRECTION * SAMPLES_PER_CALL}")
    print(f"  Total      : ~{CALLS_PER_DIRECTION * SAMPLES_PER_CALL * 2} samples")
    print(f"  Output     : {OUTPUT_FILE}")
    print()

    api_key = ''
    if not api_key:
        raise EnvironmentError("API key is required")

    client = genai.Client(api_key=api_key)
    log.info(f"Model  : {MODEL}")
    log.info(f"Output : {OUTPUT_DIR.resolve()}")
    log.info("Checkpoint system ENABLED — resumes if interrupted")

    samples = generate_boundary_samples(client)
    print_report(samples)


if __name__ == "__main__":
    main()

  Anger / Sadness Hard-Boundary Generator

  Directions : ['anger_looks_sad', 'sadness_looks_angry']
  Per dir    : 5 calls × 40 samples = 200
  Total      : ~400 samples
  Output     : /content/drive/MyDrive/emo/emotion_data/boundary_anger_sadness.json



boundary [sadness→anger] call 5: 100%|██████████| 10/10 [08:42<00:00, 52.23s/call]


  BOUNDARY SAMPLES — anger ↔ sadness  (400 total)

  Emotion split:
    anger      200  (50.0%)  ████████████████████████████████████████
    sadness    200  (50.0%)  ████████████████████████████████████████

  Target distribution:
    you          100  (25.0%)  ████████████████████
    other        101  (25.2%)  ████████████████████
    self         100  (25.0%)  ████████████████████
    situation     99  (24.8%)  ███████████████████

  Intensity  min=0.32  max=0.85  avg=0.67

  Saved to : /content/drive/MyDrive/emo/emotion_data/boundary_anger_sadness.json

  ► Merge into train.json before re-training.
    The 'is_boundary' flag lets you apply higher sample
    weights in your loss function if desired (e.g. 1.5×–2×).

